# Accounts overview

Lists every account in the active data repo with its opening balance and (if applicable) the most recent rate that the scenario applies to it.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

> **Table clipped, or the notebook slow?** Run `analysis jupyter` and open this notebook in JupyterLab: wide tables scroll sideways there, and it can use less memory than VS Code. The terminal shows the same information without a notebook: `analysis sim account list`, `analysis sim account history <account>`, `analysis sim account holdings <account>` and `analysis sim account holders <symbol>`.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.pickers import (
    scenario_picker, DollarModePicker, TimeRangePicker, ValuePlotModePicker,
    YAxisPlotModePicker,
)
from analysis.sim.notebook_ui.accounts_plots import (
    account_history_widget, show_account_setup,
)
from analysis.sim.notebook_ui.holdings_widgets import (
    account_holdings_widget, symbol_holders_widget,
)
from analysis.sim.notebook_ui.allocation_widgets import portfolio_allocation_widget
AS_OF = None  # e.g. '2026-05-31' to pin rate lookups

DOLLAR_MODE = DollarModePicker()
VALUE_MODE = ValuePlotModePicker()
Y_AXIS_MODE = YAxisPlotModePicker()
# The years the balance history shows; it starts at the whole run.
TIME_RANGE = TimeRangePicker()

# If new scenarios are added, rerun the notebook to pick them up.
# CONNECTIONS holds the live ScenarioConnection/RunConnection pair; `run` is
# None if this scenario hasn't been run yet (fine for the first two sections
# below -- only "Balance history" needs a run).
PICKER, CONNECTIONS, SCENARIO_PICKER = scenario_picker(require_run=False, default="base")
PICKER

## Balance history

Pick a **Kind** (asset, liability, income, or expense) and one or more
accounts of that kind (Cmd/Ctrl-click for multi-select) to plot their
month-end balances from the scenario's most recent run (the `latest run`
shown above).

Switching Kind repopulates the Accounts list below it, so unlike kinds (e.g.
asset vs. income) never end up plotted together on the same axis. Income and
expense balances are cumulative totals; a note appears under the chart
whenever one of those is selected.

Pick a single account with more than one lot and a **Lots** list appears
beside it: pick one or more lots to plot each lot's balance in place of the
account's. Leave it empty for the whole account.

In [ ]:
CONNECTIONS.show(account_history_widget, "asset", "Account balances",
                 scenario_picker=SCENARIO_PICKER,
                 value_mode=VALUE_MODE,
                 y_axis_mode=Y_AXIS_MODE,
                 dollar_mode=DOLLAR_MODE,
                 time_range=TIME_RANGE)

## Account setup details.

Account configuration from `accounts.csv` and `account_lots.csv`.

In [ ]:
CONNECTIONS.show_scenario(show_account_setup)

## Breakdowns per investment.

### Assets held within an account

Individual assets within each investment account.

In [ ]:
CONNECTIONS.show_scenario(account_holdings_widget)

### Accounts holding a specific asset.

In [ ]:
CONNECTIONS.show_scenario(symbol_holders_widget)

### Portfolio by asset class.

What the whole portfolio is made of, from each lot's balance and its symbol's weights in `allocations.csv`. Cash, real estate and vehicles are classed from their account; a lot whose symbol has no weights shows as *unclassified*. `analysis sim allocation` prints the same table in a terminal.

In [ ]:
CONNECTIONS.show_scenario(portfolio_allocation_widget)